## 工具

#### 包装工具调用

In [ ]:
import random

from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_deepseek import ChatDeepSeek

load_dotenv(override=True)

@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取城市的天气

    Args:
        city: 城市名称
    """
    if random.random() < 0.6:
        raise ConnectionError("获取天气失败")
    return f"{city}天气是晴朗的，温度是25摄氏度"

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

tools = [get_weather]
model_with_tools = model.bind_tools(tools)

In [ ]:
from langgraph.prebuilt.tool_node import ToolCallRequest
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from pydantic import BaseModel
from langgraph.prebuilt import ToolNode
from typing import Literal


class UserContext(BaseModel):
    max_attempts: int

def llm_node(state: MessagesState):
    ai_msg = model_with_tools.invoke(state["messages"])
    return {
        "messages": [ai_msg],
    }

def route(state: MessagesState) -> Literal["tool_node", END]:
    last_msg = state["messages"][-1]
    if isinstance(last_msg, AIMessage) and last_msg.tool_calls:
        return "tool_node"
    return END

def retry_wrap_tool_call(request: ToolCallRequest, execute):
    tool_name = request.tool_call["name"]
    max_attempts = request.runtime.context.max_attempts

    if max_attempts < 1:
        raise ValueError("max_attempts 必须至少为 1")

    for attempt in range(1, max_attempts + 1):
        try:
            return execute(request)
        except (TimeoutError, ConnectionError):
            if attempt == max_attempts:
                raise
            print(f"调用 {tool_name} 工具失败，将进行第 {attempt} 次重试")
    raise AssertionError


builder = StateGraph(MessagesState, context_schema=UserContext)
builder.add_node(llm_node)
builder.add_node("tool_node", ToolNode(tools, wrap_tool_call=retry_wrap_tool_call))
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", route)
builder.add_edge("tool_node", "llm_node")
graph = builder.compile()

result = graph.invoke({"messages": [HumanMessage("北京天气如何")]}, context=UserContext(max_attempts=3))
print(result["messages"][-1].content)